In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    KFold,
    cross_val_score
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.metrics import mean_squared_error

In [3]:
train_df = pd.read_csv(
    "../data/processed/train_after_feature_engineering.csv"
)

test_df = pd.read_csv(
    "../data/processed/test_after_feature_engineering.csv"
)

In [4]:
print(train_df.shape)

print(test_df.shape)

(1460, 158)
(1459, 157)


In [5]:
train_df.head()

,LotArea,GrLivArea,TotalBsmtSF,GarageArea,MasVnrArea,LotFrontage,FullBath,HalfBath,BedroomAbvGr,TotRmsAbvGrd,...,YrSold_2007,YrSold_2008,YrSold_2009,YrSold_2010,CentralAir,Street,HasGarage,HasBasement,HasFireplace,SalePrice
0,-0.207142,0.370333,-0.459303,0.351000,0.514104,-0.220875,0.789741,1.227585,0.163779,0.912210,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,208500
1,-0.091886,-0.482512,0.466465,-0.060731,-0.570750,0.460320,0.789741,-0.761621,0.163779,-0.318683,...,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,181500
2,0.073480,0.515013,-0.313369,0.631726,0.325915,-0.084636,0.789741,1.227585,0.163779,-0.318683,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,223500
3,-0.096897,0.383659,-0.687324,0.790804,-0.570750,-0.447940,-1.026041,-0.761621,0.163779,0.296763,...,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,140000
4,0.375148,1.299326,0.199680,1.698485,1.366489,0.641972,0.789741,1.227585,1.390023,1.527656,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,250000


In [6]:
X = train_df.drop("SalePrice", axis=1)

y = np.log1p(train_df["SalePrice"])

In [7]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [8]:
def rmse_cv(model):

    rmse = np.sqrt(
        -cross_val_score(
            model,
            X,
            y,
            scoring="neg_mean_squared_error",
            cv=kf
        )
    )

    return rmse

In [9]:
lr_model = LinearRegression()

lr_scores = rmse_cv(lr_model)

print("Linear Regression")

print("Mean RMSE:", lr_scores.mean())

print("Std RMSE:", lr_scores.std())

Linear Regression
Mean RMSE: 0.15944327476300374
Std RMSE: 0.033400148468619335


In [10]:
ridge_model = Ridge(alpha=10)

ridge_scores = rmse_cv(ridge_model)

print("Ridge")

print("Mean RMSE:", ridge_scores.mean())

print("Std RMSE:", ridge_scores.std())

Ridge
Mean RMSE: 0.15681813011385232
Std RMSE: 0.034380989776254076


In [11]:
lasso_model = Lasso(alpha=0.0005)

lasso_scores = rmse_cv(lasso_model)

print("Lasso")

print("Mean RMSE:", lasso_scores.mean())

print("Std RMSE:", lasso_scores.std())

Lasso
Mean RMSE: 0.15622136180290974
Std RMSE: 0.03522721335607219


In [12]:
elastic_model = ElasticNet(
    alpha=0.0005,
    l1_ratio=0.9
)

elastic_scores = rmse_cv(elastic_model)

print("ElasticNet")

print("Mean RMSE:", elastic_scores.mean())

print("Std RMSE:", elastic_scores.std())

ElasticNet
Mean RMSE: 0.1561616360216535
Std RMSE: 0.035046359705558125


In [13]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge",
        "Lasso",
        "ElasticNet"
    ],

    "CV RMSE Mean": [
        lr_scores.mean(),
        ridge_scores.mean(),
        lasso_scores.mean(),
        elastic_scores.mean()
    ],

    "CV RMSE Std": [
        lr_scores.std(),
        ridge_scores.std(),
        lasso_scores.std(),
        elastic_scores.std()
    ]
})

results = results.sort_values(
    by="CV RMSE Mean",
    ascending=True
)

results

,Model,CV RMSE Mean,CV RMSE Std
3,ElasticNet,0.156162,0.035046
2,Lasso,0.156221,0.035227
1,Ridge,0.156818,0.034381
0,Linear Regression,0.159443,0.033400


In [14]:
results.to_csv(
    "../model_reports/baseline_results.csv",
    index=False
)